# Часть 3. Парсинг данных из веб-сайтов и API

Пайплайн: для каждого источника берём только строки с пропусками в целевых полях,
парсим, результаты через `fillna` возвращаются в основной датасет.

1. **IMDb** -> `runtime`, `vote_average_imdb`, `vote_count_imdb`, `budget`, `genres`, `popularity_imdb_rank`
2. **Box Office Mojo** -> `revenue`, `budget`, `runtime`, `genres`
3. **TMDB API** -> `runtime`, `vote_average_imdb`, `vote_count_imdb`, `budget`, `genres`, `revenue`, `popularity_tmdb`
4. **The Numbers** -> `revenue`, `budget`


В этом ноутбуке мы загружаем подготовленный ранее датасет `netflix_enriched_2.csv`.
И обогащаем его информацией о жанрах, бюджете, выручке, хронометраже, данные об аудитории и оценке из внешних источников.

Результат этой части - это обогащенный датасет `netflix_parsed_3.csv`, который используется в последующих частях анализа.

#### Выполнили Гиллер Александр, Соломонов Егор


## Шаг 1. Импорты и загрузка датасета


In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import time
import json
import random
import logging

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By

# ошибки парсинга логируем в файл, так удобнее их разбирать https://docs.python.org/3/library/logging.html#module-logging
logging.basicConfig(
    filename="parse_errors.log",
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s %(message)s",
)

- `popularity` из обогащения - это TMDB-оценка. Переименовываем сразу, чтобы не смешивать с IMDB-рангом, который будет получен на шаге парсинга.
- Дполонительно популярность c сайта IMDB сохраним в колонку `popularity_imdb_rank`


In [94]:
df = pd.read_csv("netflix_enriched_2.csv")

df = df.rename(columns={"popularity": "popularity_tmdb"})
df["popularity_imdb_rank"] = np.nan

df[
    [
        "runtime",
        "vote_count_imdb",
        "vote_average_imdb",
        "revenue",
        "budget",
        "content_type",
        "popularity_tmdb",
        "popularity_imdb_rank",
        "genres",
    ]
].isna().sum()

runtime                 105
vote_count_imdb         285
vote_average_imdb       285
revenue                 397
budget                  397
content_type             83
popularity_tmdb         289
popularity_imdb_rank    499
genres                   83
dtype: int64

## Шаг 2. Подготовка датасета перед парсингом


### 2.1 Подготовка датасета `df`

- Приводим нужные колонки к числам
- Добавляем отсутствующие признаки


In [95]:
for col in [
    "runtime",
    "vote_count_imdb",
    "vote_average_imdb",
    "revenue",
    "budget",
    "popularity_tmdb",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "genres" not in df.columns:
    df["genres"] = np.nan

if "content_type" not in df.columns:
    df["content_type"] = np.nan

### 2.2 Вспомогательные функции

- Форматирование чисел и строк: используем регулярные выражения, так как на веб-сайтах очень часто числа отображаются в особом строковом формате, который простым преобразованием в строки в число не отформатировать


In [ ]:
def money_to_number(x):
    return float(re.sub("[^0-9]", "", str(x)))


def votes_to_number(value):
    s = str(value).strip().upper().replace(",", "")
    if "M" in s:
        return float(s.replace("M", "")) * 1000000
    if "K" in s:
        return float(s.replace("K", "")) * 1000
    return float(s)


def text_norm(text):
    return re.sub(r"\s+", " ", str(text))


def imdb_duration_to_minutes(value):
    try:
        hours = re.findall(r"(\d+)H", value)
        minutes = re.findall(r"(\d+)M", value)
        result = 0
        if hours:
            result += float(hours[0]) * 60
        if minutes:
            result += float(minutes[0])
        return result if result > 0 else np.nan
    except Exception:
        return np.nan

## Шаг 3. Парсинг с сайта [IMDB](https://www.imdb.com/)


При ручном анализе строк без `imdb_id` - 83 строки. Есть предположение, что большинство представленных тайтлов это сериалы, либо год указан некорректно. В связи с этим принято решение:

- спарсить для строк без `imdb_id` информацию напрямую с сайта imdb.com по `title` и `release_year`, где год будем проверять по точному совпадению, или попаданию в интервал (что как-раз подходит для сериалов)
- далее по ссылке с `imdb_id` парсим необходимую информацию: `runtime`, `vote_average_imdb`, `vote_count_imdb`, `budget`, `genres`, `content_type`, `popularity_imdb_rank`


In [97]:
df_no_imdb = df[df["imdb_id"].isna()]
print(df_no_imdb.shape)
df_no_imdb.head(3)

(83, 14)


,title,rating,release_year,title_lower,imdb_id,vote_average_imdb,vote_count_imdb,budget,revenue,runtime,genres,popularity_tmdb,content_type,popularity_imdb_rank
406,Grey's Anatomy,TV-14,2016,grey's anatomy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
407,Naruto,TV-PG,2008,naruto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
408,Masha and the Bear,TV-Y,2013,masha and the bear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.1 IMDB: поиск `imdb_id` по названию и году (для строк без `imdb_id`)


Подготовим метод для парсинга по `title` и `release_year` и нахождению `imdb_id`

- Проходим по всем ссылкам, чтобы получить match по году и названию
- возможный формат года: 2012, 2012-, 2012-2015, проверяем через регулярное выражение
- если нет совпадений по году - берем первый элемент на странице (fallback)


In [ ]:
def find_imdb_id_by_search(title, year):
    query = f"{title}"
    url = f"https://www.imdb.com/find/?q={query}&s=tt"

    def scan_results(soup):
        section = soup.select_one('[data-testid="find-results-section-title"]')
        if not section:
            return None

        ul = section.find("ul")
        if not ul:
            return None

        for li in ul.find_all("li", recursive=False):
            link = li.select_one('a[href*="/title/tt"]')
            if not link:
                continue

            match = re.search(r"/title/(tt\d+)/", link.get("href", ""))
            if not match:
                continue

            found_id = match.group(1)
            ym = re.search(r"(\d{4})\s*([\u2013\u2014\-]\s*(\d{4})?)?", li.get_text())
            if ym:
                start = int(ym.group(1))
                has_dash = ym.group(2) is not None
                end = int(ym.group(3)) if ym.group(3) else None

                if not has_dash:
                    year_ok = start == int(year)
                elif end is None:
                    year_ok = int(year) >= start
                else:
                    year_ok = start <= int(year) <= end

                if year_ok:
                    return found_id
        return None

    try:
        driver.get(url)
        time.sleep(random.uniform(1.5, 3))

        # Нажимаем "More popular matches" если кнопка есть
        try:
            more_btn = driver.find_element(
                By.CSS_SELECTOR, "button.ipc-see-more__button"
            )
            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(random.uniform(1.5, 3))
            soup = BeautifulSoup(driver.page_source, "html.parser")
            found_id = scan_results(soup)
            if found_id:
                return found_id
        except Exception:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")
        found_id = scan_results(soup)
        if found_id:
            return found_id

        # Fallback: первый элемент без проверки года
        section = soup.select_one('[data-testid="find-results-section-title"]')
        if section:
            ul = section.find("ul")
            if ul:
                first_li = ul.find("li")
                if first_li:
                    link = first_li.select_one('a[href*="/title/tt"]')
                    if link:
                        match = re.search(r"/title/(tt\d+)/", link.get("href", ""))
                        if match:
                            print(
                                f'[IMDb search] "{title}" ({year}) -> {match.group(1)} (fallback)'
                            )
                            return match.group(1)

        print(f'[IMDb search] "{title}" ({year}) не найдено')

    except Exception as e:
        logging.warning(f"find_imdb_id_by_search [{title} {year}]: {e}")

    return None

### 3.2 IMDB: парсинг текущей страницы

Прасим страницу полученную по `imdb_id` и находим интересующую нас информацию: `runtime`, `vote_average_imdb`, `vote_count_imdb`, `budget`, `genres`, `content_type`, `popularity_imdb_rank`


In [102]:
IMDB_CONTENT_TYPE_MAP = {
    "Movie": "movie",
    "TVSeries": "tvSeries",
    "TVMiniSeries": "tvMiniSeries",
    "TVMovie": "tvMovie",
    "TVEpisode": "tvEpisode",
    "TVSpecial": "tvSpecial",
    "Short": "short",
    "Video": "video",
    "VideoGame": "videoGame",
}


def parse_imdb_page():
    parsed_runtime = np.nan
    parsed_vote_average = np.nan
    parsed_vote_count = np.nan
    parsed_budget = np.nan
    parsed_genres = np.nan
    parsed_content_type = np.nan
    parsed_popularity = np.nan

    try:
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        for script in soup.find_all("script", {"type": "application/ld+json"}):
            try:
                data = json.loads(script.string)
                if isinstance(data, list):
                    data = data[0]
                if "@type" in data:
                    parsed_content_type = IMDB_CONTENT_TYPE_MAP.get(
                        data["@type"], data["@type"]
                    )
                if "duration" in data:
                    parsed_runtime = imdb_duration_to_minutes(data["duration"])
                if "aggregateRating" in data:
                    parsed_vote_average = float(data["aggregateRating"]["ratingValue"])
                    parsed_vote_count = float(data["aggregateRating"]["ratingCount"])
                if "genre" in data:
                    g = data["genre"]
                    parsed_genres = ", ".join(g) if isinstance(g, list) else str(g)
            except Exception:
                pass

    except Exception as e:
        logging.warning(f"parse_imdb_page LD+JSON: {e}")

    try:
        text = text_norm(driver.find_element(By.TAG_NAME, "body").text)

        if pd.isna(parsed_runtime):
            match_1 = re.findall(r"Runtime (\d+) hours? (\d+) minutes?", text, re.I)
            match_2 = re.findall(r"Runtime (\d+) minutes?", text, re.I)
            if match_1:
                parsed_runtime = float(match_1[0][0]) * 60 + float(match_1[0][1])
            elif match_2:
                parsed_runtime = float(match_2[0])

        if pd.isna(parsed_vote_average):
            match_rating = re.findall(
                r"IMDb RATING\s+([0-9.]+)\s*/\s*10\s+([0-9.,KMk]+)", text, re.I
            )
            if not match_rating:
                match_rating = re.findall(
                    r"([0-9.]+)\s*/\s*10\s+([0-9.,KMk]+)", text, re.I
                )
            if match_rating:
                parsed_vote_average = float(match_rating[0][0])
                try:
                    parsed_vote_count = votes_to_number(match_rating[0][1])
                except Exception:
                    pass

        match_budget = re.findall(r"Budget\s*\$[0-9,]+", text, re.I)
        if match_budget:
            parsed_budget = money_to_number(match_budget[0])

        if pd.isna(parsed_popularity):
            match_popularity = re.findall(r"Popularity\s*#?\s*([\d,]+)", text, re.I)
            if match_popularity:
                parsed_popularity = float(match_popularity[0].replace(",", ""))

    except Exception as e:
        logging.warning(f"parse_imdb_page text fallback: {e}")

    return pd.Series(
        [
            parsed_runtime,
            parsed_vote_average,
            parsed_vote_count,
            parsed_budget,
            parsed_genres,
            parsed_content_type,
            parsed_popularity,
        ]
    )


### 3.3 IMDB: единая точка входа

Если `imdb_id` есть - идём напрямую по URL. Если нет - ищем по названию + году через метод `find_imdb_id_by_search`, сохраняем найденный `imdb_id` в датасет.


In [ ]:
IMDB_PARSE_COLS = [
    "p_runtime",
    "p_vote_avg",
    "p_vote_cnt",
    "p_budget",
    "p_genres",
    "p_content_type",
    "p_popularity",
    "p_imdb_id",
]
EMPTY_IMDB_COLS = pd.Series([np.nan] * 8, index=IMDB_PARSE_COLS)


def parse_imdb(row):
    title = row["title"]
    year = int(row["release_year"])
    found_imdb_id = row["imdb_id"] if pd.notna(row.get("imdb_id", np.nan)) else None

    try:
        if found_imdb_id:
            url = f"https://www.imdb.com/title/{found_imdb_id}/"
            driver.get(url)
            time.sleep(random.uniform(2, 3))
        else:
            found_imdb_id = find_imdb_id_by_search(title, year)
            if found_imdb_id is None:
                print(f'[IMDB] "{title}" ({year}) -> imdb_id не найден')
                return EMPTY_IMDB_COLS
            url = f"https://www.imdb.com/title/{found_imdb_id}/"
            driver.get(url)
            time.sleep(random.uniform(2, 3))

    except Exception as e:
        logging.warning(f"parse_imdb navigate [{title}]: {e}")
        return EMPTY_IMDB_COLS

    result = parse_imdb_page()
    result.index = IMDB_PARSE_COLS[:-1]
    result["p_imdb_id"] = found_imdb_id

    return result


### 3.4 Запускаем Selenium


In [104]:
driver = webdriver.Chrome()
driver.set_page_load_timeout(40)

### 3.5 Парсинг IMDB

Берём только строки с хотя бы одним пропуском в целевых полях.
Результаты через `fillna` возвращаются в `df`. Найденный `imdb_id` тоже сохраняется.


In [ ]:
IMDB_FIELDS = [
    "runtime",
    "vote_average_imdb",
    "vote_count_imdb",
    "budget",
    "genres",
    "content_type",
    "popularity_imdb_rank",
    "imdb_id",
]
STEP = 50

imdb_mask = df[IMDB_FIELDS].isna().any(axis=1)
imdb_todo = df[imdb_mask].copy()
print(f"Строк для IMDb: {len(imdb_todo)}")

tqdm.pandas(desc="Парсим IMDB")

for start in range(0, len(imdb_todo), STEP):
    chunk = imdb_todo.iloc[start : start + STEP]

    results = chunk.progress_apply(parse_imdb, axis=1)
    results.columns = IMDB_PARSE_COLS

    for field, pcol in zip(IMDB_FIELDS, IMDB_PARSE_COLS[:-1]):
        df.loc[chunk.index, field] = df.loc[chunk.index, field].fillna(results[pcol])

    no_id_mask = df.loc[chunk.index, "imdb_id"].isna()
    if no_id_mask.any():
        found_ids = results.loc[chunk.index[no_id_mask], "p_imdb_id"].dropna()
        if not found_ids.empty:
            df.loc[found_ids.index, "imdb_id"] = found_ids.values

    df.to_csv("netflix_parsed_imdb.csv", index=False)

print("\nIMDB готово. Пропуски:")
print(df[IMDB_FIELDS].isna().sum())

Строк для IMDb: 499


Парсим IMDb:   4%|▍         | 2/49 [00:12<04:46,  6.09s/it]

[IMDb search] "LeapFrog: The Amazing Alphabet Amusement Park" (2010) -> tt3725800 (fallback)


Парсим IMDb:   8%|▊         | 4/49 [00:35<07:20,  9.80s/it]

[IMDb search] "Mighty Morphin Power Rangers" (2010) -> tt0106064 (fallback)


...

[IMDb search] "DreamWorks Spooky Stories: Volume 2" (2011) -> tt3608838 (fallback)


Парсим IMDb:  76%|███████▌  | 37/49 [06:23<01:52,  9.41s/it]

[IMDb search] "Back to the Secret Garden" (2002) -> tt0233277 (fallback)


Парсим IMDb: 100%|██████████| 49/49 [08:29<00:00, 10.41s/it]


IMDb готово. Пропуски:
runtime                  77
vote_average_imdb        24
vote_count_imdb          24
budget                  375
genres                    4
content_type              0
popularity_imdb_rank    273
imdb_id                   0
dtype: int64


In [106]:
driver.quit()

In [107]:
df.isna().sum()

title                     0
rating                    0
release_year              0
title_lower               0
imdb_id                   0
vote_average_imdb        24
vote_count_imdb          24
budget                  375
revenue                 397
runtime                  77
genres                    4
popularity_tmdb         289
content_type              0
popularity_imdb_rank    273
dtype: int64

## Шаг 4. Парсинг с сайта [Box Office Mojo](https://www.boxofficemojo.com/)

На предыдущем шаге нам удалось значительно сократить пропуски по признакам `vote_average_imdb`, `vote_count_imdb`, `runtime`, `genres`. Но все еще много пропусков по `revenue` и `budget`.

На данном шаге выполним парсинг по `imdb_id` для заполнения `runtime`, `genres`, `revenue` и `budget`


### 4.1 Box Office Mojo: единая точка входа

По `imdb_id` идём напрямую по URL и парсим данные


In [108]:
def parse_mojo(row):
    imdb_id = row["imdb_id"]
    url = f"https://www.boxofficemojo.com/title/{imdb_id}/"
    parsed_revenue = np.nan
    parsed_budget = np.nan
    parsed_runtime = np.nan
    parsed_genres = np.nan

    try:
        driver.get(url)
        time.sleep(random.uniform(2, 3))

        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        block = soup.select_one(".mojo-performance-summary-table")
        if block is not None:
            match_revenue = re.findall(
                r"Worldwide\s*(\$[0-9,]+)", block.get_text(" ", strip=True), re.I
            )
            if match_revenue:
                parsed_revenue = money_to_number(match_revenue[0])

        text = text_norm(driver.find_element(By.TAG_NAME, "body").text)

        if pd.isna(parsed_revenue):
            match_revenue = re.findall(r"Worldwide\s*\$[0-9,]+", text, re.I)
            if match_revenue:
                parsed_revenue = money_to_number(match_revenue[0])

        match_budget = re.findall(r"Budget\s*\$[0-9,]+", text, re.I)
        if match_budget:
            parsed_budget = money_to_number(match_budget[0])

        match_runtime = re.findall(
            r"Running\s+Time\s+(\d+)\s+hr\s+(\d+)\s+min", text, re.I
        )
        if match_runtime:
            parsed_runtime = float(match_runtime[0][0]) * 60 + float(
                match_runtime[0][1]
            )
        else:
            match_runtime = re.findall(r"Running\s+Time\s+(\d+)\s+min", text, re.I)
            if match_runtime:
                parsed_runtime = float(match_runtime[0])

        genres_label = soup.find("span", string=re.compile(r"^Genres$", re.I))
        if genres_label:
            genres_span = genres_label.find_next_sibling("span")
            if genres_span:
                raw = genres_span.get_text(strip=True)
                parsed_genres = ", ".join(raw.split())

    except Exception as e:
        logging.warning(f"parse_mojo [{imdb_id}]: {e}")

    found_parts = []
    if pd.notna(parsed_revenue):
        found_parts.append(f"revenue={parsed_revenue:.0f}")
    if pd.notna(parsed_budget):
        found_parts.append(f"budget={parsed_budget:.0f}")
    if pd.notna(parsed_runtime):
        found_parts.append(f"runtime={parsed_runtime:.0f}")
    if pd.notna(parsed_genres):
        found_parts.append(f"genres={parsed_genres}")

    return pd.Series([parsed_revenue, parsed_budget, parsed_runtime, parsed_genres])


### 4.2 Запускаем Selenium


In [109]:
driver = webdriver.Chrome()
driver.set_page_load_timeout(40)

### 4.3 Парсинг Box Office Mojo

Фильтруем строки с пропусками в `revenue`, `budget`, `runtime`, `genres` и с известным `imdb_id`.


In [110]:
MOJO_FIELDS = ["revenue", "budget", "runtime", "genres"]

mojo_mask = df[MOJO_FIELDS].isna().any(axis=1) & df["imdb_id"].notna()
mojo_todo = df[mojo_mask].copy()
print(f"Строк для Mojo: {len(mojo_todo)}")

tqdm.pandas(desc="Парсим Box Office Mojo")

for start in range(0, len(mojo_todo), STEP):
    chunk = mojo_todo.iloc[start : start + STEP]

    results = chunk.progress_apply(parse_mojo, axis=1)
    results.columns = ["p_revenue", "p_budget", "p_runtime", "p_genres"]

    for field, pcol in zip(
        MOJO_FIELDS, ["p_revenue", "p_budget", "p_runtime", "p_genres"]
    ):
        df.loc[chunk.index, field] = df.loc[chunk.index, field].fillna(results[pcol])

    df.to_csv("netflix_parsed_mojo.csv", index=False)

print("\nMojo готово. Пропуски:")
print(df[MOJO_FIELDS].isna().sum())

Строк для Mojo: 406


Парсим Box Office Mojo: 100%|██████████| 6/6 [00:23<00:00,  3.93s/it]


Mojo готово. Пропуски:
revenue    353
budget     375
runtime     33
genres       4
dtype: int64


In [111]:
driver.quit()

In [112]:
print(df.isna().sum())

title                     0
rating                    0
release_year              0
title_lower               0
imdb_id                   0
vote_average_imdb        24
vote_count_imdb          24
budget                  375
revenue                 353
runtime                  33
genres                    4
popularity_tmdb         289
content_type              0
popularity_imdb_rank    273
dtype: int64


## Шаг 5. Парсинг через API TMDB

На предыдущем шаге нам удалось значительно сократить пропуски по признакам `runtime`.

Для строк с оставшимися пропусками в `runtime`, `vote_average_imdb`, `vote_count_imdb`, `budget`, `genres`, `revenue`, `popularity_tmdb` используем TMDB API:

1. `/find/{imdb_id}?external_source=imdb_id` - получаем TMDB `id` и тип (`movie` / `tv`)
2. `/movie/{id}` или `/tv/{id}` - берём нужные поля

TMDB API заполняет `popularity_tmdb`, тот же источник, что и исходный датасет TMDB

API работает без Selenium, TMDB_BEARER token был получен через регистрацию на сайте (удален из исходного кода в целях конфиденциальности). Для парсинга нужен VPN, так как API не доступно в России.


In [ ]:
TMDB_BEARER = "TOKEN"

TMDB_HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_BEARER}",
}


TMDB_FIELDS = [
    "runtime",
    "vote_average_imdb",
    "vote_count_imdb",
    "budget",
    "genres",
    "revenue",
    "popularity_tmdb",
]
TMDB_PARSE_COLS = [
    "p_runtime",
    "p_vote_avg",
    "p_vote_cnt",
    "p_budget",
    "p_genres",
    "p_revenue",
    "p_popularity",
]
EMPTY_TMDB = pd.Series([np.nan] * len(TMDB_PARSE_COLS), index=TMDB_PARSE_COLS)


def fetch_tmdb(imdb_id):
    # Шаг 1: /find/{tmdb_id} + {media_type}
    try:
        r = requests.get(
            f"https://api.themoviedb.org/3/find/{imdb_id}",
            headers=TMDB_HEADERS,
            params={"external_source": "imdb_id", "language": "en-US"},
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        logging.warning(f"fetch_tmdb find [{imdb_id}]: {e}")
        return EMPTY_TMDB.copy()

    tmdb_id = None
    media_type = None
    for key, mtype in [
        ("movie_results", "movie"),
        ("tv_results", "tv"),
        ("tv_episode_results", "tv_episode"),
    ]:
        results = data.get(key, [])
        if results:
            tmdb_id = results[0]["id"]
            media_type = mtype
            break

    if tmdb_id is None:
        return EMPTY_TMDB.copy()

    # Шаг 2: /movie/{tmdb_id} или /tv/{tmdb_id}
    detail_url = (
        f"https://api.themoviedb.org/3/movie/{tmdb_id}"
        if media_type == "movie"
        else f"https://api.themoviedb.org/3/tv/{tmdb_id}"
    )
    try:
        r = requests.get(
            detail_url,
            headers=TMDB_HEADERS,
            params={"language": "en-US"},
            timeout=10,
        )
        r.raise_for_status()
        d = r.json()
    except Exception as e:
        logging.warning(f"fetch_tmdb detail [{imdb_id} -> {tmdb_id}]: {e}")
        return EMPTY_TMDB.copy()

    genres_list = []
    for g in d.get("genres", []):
        if g["name"].count(" & "):
            g_list = g["name"].split(" & ")
            genres_list.extend(g_list)
        else:
            genres_list.append(g["name"])

    parsed_genres = ", ".join(genres_list) if genres_list else np.nan

    if media_type == "movie":
        parsed_runtime = float(d["runtime"]) if d.get("runtime") else np.nan
        parsed_budget = float(d["budget"]) if d.get("budget") else np.nan
        parsed_revenue = float(d["revenue"]) if d.get("revenue") else np.nan
    else:
        run_times = d.get("episode_run_time", [])
        parsed_runtime = float(run_times[0]) if run_times else np.nan
        parsed_budget = np.nan
        parsed_revenue = np.nan

    parsed_popularity = float(d["popularity"]) if d.get("popularity") else np.nan
    parsed_vote_avg = float(d["vote_average"]) if d.get("vote_average") else np.nan
    parsed_vote_cnt = float(d["vote_count"]) if d.get("vote_count") else np.nan

    return pd.Series(
        [
            parsed_runtime,
            parsed_vote_avg,
            parsed_vote_cnt,
            parsed_budget,
            parsed_genres,
            parsed_revenue,
            parsed_popularity,
        ],
        index=TMDB_PARSE_COLS,
    )

In [139]:
tmdb_mask = df[TMDB_FIELDS].isna().any(axis=1) & df["imdb_id"].notna()
tmdb_todo = df[tmdb_mask].copy()
print(f"Строк для TMDB: {len(tmdb_todo)}")

for idx, row in tqdm(tmdb_todo.iterrows(), total=len(tmdb_todo), desc="TMDB API"):
    result = fetch_tmdb(row["imdb_id"])
    for field, pcol in zip(TMDB_FIELDS, TMDB_PARSE_COLS):
        if pd.isna(df.loc[idx, field]) and pd.notna(result[pcol]):
            df.loc[idx, field] = result[pcol]
    time.sleep(0.25)

df.to_csv("netflix_parsed_tmdb.csv", index=False)
print("\nTMDB готово. Пропуски:")
print(df[TMDB_FIELDS].isna().sum())

Строк для TMDB: 395


TMDB API: 100%|██████████| 395/395 [05:54<00:00,  1.11it/s]


TMDB готово. Пропуски:
runtime               24
vote_average_imdb     24
vote_count_imdb       24
budget               369
genres                 2
revenue              353
popularity_tmdb       36
dtype: int64


In [140]:
df.isna().sum()

title                     0
rating                    0
release_year              0
title_lower               0
imdb_id                   0
vote_average_imdb        24
vote_count_imdb          24
budget                  369
revenue                 353
runtime                  24
genres                    2
popularity_tmdb          36
content_type              0
popularity_imdb_rank    273
dtype: int64

## Шаг 6. Парсинг с сайта [The Numbers](https://www.the-numbers.com/)

На предыдущем шаге нам удалось значительно сократить пропуски по признакам `runtime`, `popularity_tmdb`. Но все еще много пропусков по `revenue` и `budget`.

На данном шаге выполним парсинг по `title` и `release_year` для заполнения `revenue` и `budget`


### 6.1 Вспомогательные функции


Подготовка url для поиска


In [141]:
def make_numbers_title(title):
    title = str(title)
    for ch, repl in [
        (": ", "-"),
        (":", ""),
        (",", ""),
        (".", ""),
        ("'", ""),
        ("&", "and"),
        ("/", "-"),
        ("(", ""),
        (")", ""),
        ("!", ""),
        ("?", ""),
        ("#", ""),
    ]:
        title = title.replace(ch, repl)
    words = title.split()
    if words and words[0] == "The":
        words = words[1:] + [words[0]]
    return "-".join(words)


def make_numbers_url_without_year(title):
    link = f"https://www.the-numbers.com/movie/{make_numbers_title(title)}#more"
    return link


def make_numbers_url_with_year(title, year):
    link = f"https://www.the-numbers.com/movie/{make_numbers_title(title)}-({int(year)})#more"
    return link


def make_numbers_search_url(title, year):
    query = f"{title} {int(year)}".replace(" ", "+")
    link = f"https://www.the-numbers.com/search?searchterm={query}"
    return link

### 6.2 The Numbers: опсновные функции для парсинга страниц и сбора данных


In [142]:
def parse_numbers_current_page():
    parsed_budget = np.nan
    parsed_revenue = np.nan

    try:
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        table = soup.select_one("#movie_finances")
        if table is not None:
            for row in table.find_all("tr"):
                cells = [c.get_text(" ", strip=True) for c in row.find_all("td")]
                if (
                    len(cells) > 1
                    and "Worldwide Box Office" in cells[0]
                    and "$" in cells[1]
                ):
                    parsed_revenue = money_to_number(cells[1])

        for row in soup.find_all("tr"):
            row_text = row.get_text(" ", strip=True)
            if "Production Budget" in row_text:
                m = re.findall(r"\$[0-9,]+", row_text)
                if m:
                    parsed_budget = money_to_number(m[0])

        text = text_norm(soup.get_text(" ", strip=True))

        if pd.isna(parsed_budget):
            m = re.findall(r"Production\s+Budget:?\s*(\$[0-9,]+)", text, re.I)
            if m:
                parsed_budget = money_to_number(m[0])

        if pd.isna(parsed_revenue):
            m = re.findall(r"Worldwide\s+Box\s+Office\s*(\$[0-9,]+)", text, re.I)
            if m:
                parsed_revenue = money_to_number(m[0])

    except Exception as e:
        logging.warning(f"parse_numbers_current_page: {e}")

    return pd.Series([parsed_budget, parsed_revenue])


def parse_numbers_page(url):
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 3))
        return parse_numbers_current_page()
    except Exception:
        try:
            driver.execute_script("window.stop();")
            time.sleep(1)
            return parse_numbers_current_page()
        except Exception as e:
            logging.warning(f"parse_numbers_page [{url}]: {e}")
            return pd.Series([np.nan, np.nan])


def parse_numbers_search(title, year):
    url = make_numbers_search_url(title, year)
    try:
        driver.get(url)
        time.sleep(random.uniform(2.5, 4.0))

        links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/movie/']")
        title_words = set(str(title).lower().split())
        stop_words = {"the", "a", "an", "of", "in", "and"}
        meaningful = title_words - stop_words

        for link in links:
            href = link.get_attribute("href")
            link_text = str(link.text).lower()
            if href and sum(1 for w in meaningful if w in link_text) >= min(
                2, len(meaningful)
            ):
                return parse_numbers_page(href)

    except Exception as e:
        logging.warning(f"parse_numbers_search [{title} {year}]: {e}")

    return pd.Series([np.nan, np.nan])


def parse_numbers(title, year):
    url1 = make_numbers_url_without_year(title)
    result = parse_numbers_page(url1)

    if result.isna().all():
        url2 = make_numbers_url_with_year(title, year)
        result = parse_numbers_page(url2)

    if result.isna().all():
        result = parse_numbers_search(title, year)

    found_parts = []
    if pd.notna(result.iloc[0]):
        found_parts.append(f"budget={result.iloc[0]:.0f}")
    if pd.notna(result.iloc[1]):
        found_parts.append(f"revenue={result.iloc[1]:.0f}")

    return result

### 6.3 Запускаем Selenium


In [143]:
driver = webdriver.Chrome()
driver.set_page_load_timeout(40)

### 6.4 Парсинг The Numbers

Фильтруем строки с пропусками в `revenue` или `budget`.


In [144]:
NUMBERS_FIELDS = ["revenue", "budget"]

numbers_mask = df[NUMBERS_FIELDS].isna().any(axis=1)
numbers_todo = df[numbers_mask].copy()
print(f"Строк для The Numbers: {len(numbers_todo)}")

tqdm.pandas(desc="Парсим The Numbers")

for start in range(0, len(numbers_todo), STEP):
    chunk = numbers_todo.iloc[start : start + STEP]

    results = chunk.progress_apply(
        lambda x: parse_numbers(x["title"], x["release_year"]), axis=1
    )
    results.columns = ["p_budget", "p_revenue"]

    df.loc[chunk.index, "budget"] = df.loc[chunk.index, "budget"].fillna(
        results["p_budget"]
    )
    df.loc[chunk.index, "revenue"] = df.loc[chunk.index, "revenue"].fillna(
        results["p_revenue"]
    )

    df.to_csv("netflix_parsed_numbers.csv", index=False)

print("\nThe Numbers готово. Пропуски:")
print(df[NUMBERS_FIELDS].isna().sum())


Строк для The Numbers: 389


Парсим The Numbers: 100%|██████████| 39/39 [06:46<00:00, 10.43s/it]


The Numbers готово. Пропуски:
revenue    334
budget     350
dtype: int64


### Закрываем браузер и сохраняем итоговый датасет


In [145]:
driver.quit()

In [146]:
df.to_csv("netflix_parsed_3.csv", index=False)

print("Итоговые пропуски:")
print(
    df[
        [
            "runtime",
            "vote_average_imdb",
            "vote_count_imdb",
            "budget",
            "revenue",
            "genres",
            "content_type",
            "popularity_tmdb",
            "popularity_imdb_rank",
        ]
    ]
    .isna()
    .sum()
)

Итоговые пропуски:
runtime                  24
vote_average_imdb        24
vote_count_imdb          24
budget                  350
revenue                 334
genres                    2
content_type              0
popularity_tmdb          36
popularity_imdb_rank    273
dtype: int64
